In [2]:
import pandas as pd
import numpy as np  
import matplotlib.pyplot as plt
import seaborn as sb
import plotly.express as px

pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)

In [3]:
a=pd.read_csv("cleaned_train_data.csv")
b=pd.read_csv("cleaned_test_data.csv")

In [22]:
a.head()

,cc_num,merchant,category,amt,first,last,gender,street,city,state,zip,lat,long,city_pop,job,trans_num,unix_time,merch_lat,merch_long,is_fraud,age,date,time
0,2703186189652095,"fraud_Rippin, Kub and Mann",misc_net,4.97,Jennifer,Banks,F,561 Perry Cove,Moravian Falls,NC,28654,36.0788,-81.1781,3495,"Psychologist, counselling",0b242abb623afc578575680df30655b9,1325376018,36.011293,-82.048315,0,37,2019-01-01,00:00:18
1,630423337322,"fraud_Heller, Gutmann and Zieme",grocery_pos,107.23,Stephanie,Gill,F,43039 Riley Greens Suite 393,Orient,WA,99160,48.8878,-118.2105,149,Special educational needs teacher,1f76529f8574734946361c461b024d99,1325376044,49.159047,-118.186462,0,47,2019-01-01,00:00:44
2,38859492057661,fraud_Lind-Buckridge,entertainment,220.11,Edward,Sanchez,M,594 White Dale Suite 530,Malad City,ID,83252,42.1808,-112.2620,4154,Nature conservation officer,a1a22d70485983eac12b5b88dad1cf95,1325376051,43.150704,-112.154481,0,63,2019-01-01,00:00:51
3,3534093764340240,"fraud_Kutch, Hermiston and Farrell",gas_transport,45.00,Jeremy,White,M,9443 Cynthia Court Apt. 038,Boulder,MT,59632,46.2306,-112.1138,1939,Patent attorney,6b849c168bdad6f867558c3793159a81,1325376076,47.034331,-112.561071,0,58,2019-01-01,00:01:16
4,375534208663984,fraud_Keeling-Crist,misc_pos,41.96,Tyler,Garcia,M,408 Bradley Rest,Doe Hill,VA,24433,38.4207,-79.4629,99,Dance movement psychotherapist,a41d7549acf90789359a9aa5346dcb46,1325376186,38.674999,-78.632459,0,39,2019-01-01,00:03:06


In [4]:
df1 = a.drop(columns=["trans_num","first","last","street","zip","unix_time","merchant","lat","long","merch_lat","merch_long","cc_num","state"],axis=1)
df2 = b.drop(columns=["trans_num","first","last","street","zip","unix_time","merchant","lat","long","merch_lat","merch_long","cc_num","state"],axis=1)


In [5]:
df1.head()

,category,amt,gender,city,city_pop,job,is_fraud,age,date,time
0,misc_net,4.97,F,Moravian Falls,3495,"Psychologist, counselling",0,37,2019-01-01,00:00:18
1,grocery_pos,107.23,F,Orient,149,Special educational needs teacher,0,47,2019-01-01,00:00:44
2,entertainment,220.11,M,Malad City,4154,Nature conservation officer,0,63,2019-01-01,00:00:51
3,gas_transport,45.00,M,Boulder,1939,Patent attorney,0,58,2019-01-01,00:01:16
4,misc_pos,41.96,M,Doe Hill,99,Dance movement psychotherapist,0,39,2019-01-01,00:03:06


In [27]:
# Target encode 'city' and 'job', then label encode 'category' and 'gender'
from category_encoders import TargetEncoder
from sklearn.preprocessing import LabelEncoder

y = df1['is_fraud']
X = df1.drop('is_fraud', axis=1)
y_test = df2['is_fraud']
X_test = df2.drop('is_fraud', axis=1)

# Target encoding for 'city' and 'job'
te = TargetEncoder(cols=['city', 'job'])
X = te.fit_transform(X, y)
X_test = te.transform(X_test)

# Label encoding for specific columns
for col in ['category', 'gender']:
    le = LabelEncoder()
    X[col] = le.fit_transform(X[col].astype(str))
    X_test[col] = le.transform(X_test[col].astype(str))

print(X.head())

   category     amt  gender      city  city_pop       job  age        date  \
0         8    4.97       0  0.000000      3495  0.001693   37  2019-01-01   
1         4  107.23       0  0.000000       149  0.002157   47  2019-01-01   
2         0  220.11       1  0.000000      4154  0.015656   63  2019-01-01   
3         2   45.00       1  0.030426      1939  0.007905   58  2019-01-01   
4         9   41.96       1  0.000000        99  0.000000   39  2019-01-01   

       time  
0  00:00:18  
1  00:00:44  
2  00:00:51  
3  00:01:16  
4  00:03:06  


In [28]:
X.drop(['date', 'time'], axis=1, inplace=True)
X_test.drop(['date', 'time'], axis=1, inplace=True)

In [29]:
X.head()

,category,amt,gender,city,city_pop,job,age
0,8,4.97,0,0.000000,3495,0.001693,37
1,4,107.23,0,0.000000,149,0.002157,47
2,0,220.11,1,0.000000,4154,0.015656,63
3,2,45.00,1,0.030426,1939,0.007905,58
4,9,41.96,1,0.000000,99,0.000000,39


In [31]:
# Drop 'datetime' column before SMOTE if not needed for modeling
X_for_smote = X.copy()
X_test_for_smote = X_test.copy()

from imblearn.over_sampling import SMOTE
target_smote = SMOTE(random_state=42)
X_bal, y_bal = target_smote.fit_resample(X_for_smote, y)
X_test_bal, y_test_bal = target_smote.fit_resample(X_test_for_smote, y_test)
print('Class distribution after SMOTE:')
print(y_bal.value_counts())

c:\Users\admin\anaconda3\Lib\site-packages\sklearn\base.py:474: FutureWarning: `BaseEstimator._validate_data` is deprecated in 1.6 and will be removed in 1.7. Use `sklearn.utils.validation.validate_data` instead. This function becomes public and is part of the scikit-learn developer API.
  warnings.warn(
c:\Users\admin\anaconda3\Lib\site-packages\sklearn\base.py:474: FutureWarning: `BaseEstimator._validate_data` is deprecated in 1.6 and will be removed in 1.7. Use `sklearn.utils.validation.validate_data` instead. This function becomes public and is part of the scikit-learn developer API.
  warnings.warn(


Class distribution after SMOTE:
is_fraud
0    1289169
1    1289169
Name: count, dtype: int64


In [18]:
X_bal.shape

(2578338, 7)

In [33]:
# Standardize features after SMOTE
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_bal)
X_test_scaled = scaler.fit_transform(X_test_bal)
print('Shape after scaling:', X_scaled.shape)

Shape after scaling: (2578338, 7)


In [34]:
X_scaled=pd.DataFrame(X_scaled, columns=X_bal.columns)
X_test_scaled=pd.DataFrame(X_test_scaled, columns=X_test_bal.columns)

In [35]:
X_scaled.head()

,category,amt,gender,city,city_pop,job,age
0,0.366136,-0.786597,-0.835353,-0.358220,-0.284721,-0.278714,-0.898378
1,-0.703894,-0.513190,-0.835353,-0.358220,-0.295376,-0.265319,-0.324796
2,-1.773924,-0.211388,1.197098,-0.358220,-0.282623,0.123723,0.592936
3,-1.238909,-0.679571,1.197098,0.235828,-0.289676,-0.099657,0.306145
4,0.633644,-0.687699,1.197098,-0.358220,-0.295535,-0.327496,-0.783662


In [36]:
X_test_scaled.head()

,category,amt,gender,city,city_pop,job,age
0,0.903525,-0.790996,1.226563,0.297933,0.973931,1.002107,0.277949
1,0.903525,-0.718832,-0.815286,0.223492,-0.289368,0.550960,-1.023114
2,-0.451026,-0.688234,-0.815286,-0.077517,-0.159723,-0.439210,0.100531
3,0.632615,-0.638029,1.226563,-0.282675,-0.082866,1.897552,-0.845696
4,1.716255,-0.790114,1.226563,0.125782,-0.286244,1.273231,1.046759


In [37]:
y_test_scaled = y_test_bal  
y_scaled=y_bal

In [38]:
df_train=pd.concat([X_scaled, y_scaled], axis=1)
df_test=pd.concat([X_test_scaled, y_test_scaled], axis=1)

In [40]:
df_train.to_csv('train_preprocessed.csv', index=False)
df_test.to_csv('test_preprocessed.csv', index=False)